<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/I_want_you_to_create_the_commands_in_collab_to_ru_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this framework in a Google Colab environment, you need to execute the following commands in sequence. These commands set up the Python backend to handle the **Vector Flow** data, initialize the **Blockchain** ledger, and render the **HTML5** drawing interface.

### Step 1: Initialize the Blockchain Ledger

Copy and run this code in the first cell. It sets up the cryptographic hashing and the list that will store your immutable vector data.

In [3]:
import hashlib
import time
import json
import IPython
from google.colab import output

# Initialize the Vector Flow Blockchain
vector_blockchain = []

def commit_vector_flow(vector_data_json):
    """
    Translates raw input coordinates into a hashed vector block.
    This acts as the computer's bridge between the pen and the ledger.
    """
    path_data = json.loads(vector_data_json)

    # Identify the previous link in the chain
    prev_hash = vector_blockchain[-1]['hash'] if vector_blockchain else "0" * 64
    timestamp = time.time()

    # Construct the block payload for hashing
    block_content = f"{prev_hash}{timestamp}{vector_data_json}"
    current_hash = hashlib.sha256(block_content.encode()).hexdigest()

    block = {
        'index': len(vector_blockchain),
        'timestamp': timestamp,
        'vector_points': len(path_data),
        'previous_hash': prev_hash,
        'hash': current_hash,
        'vector_data': path_data
    }

    vector_blockchain.append(block)
    print(f"\n[VECTOR FLOW BLOCK {block['index']} COMMITTED]")
    print(f"Cryptographic Signature: {current_hash}")
    return block

# Register the Python function so the Drawing Pad can send data here
output.register_callback('notebook.commit_vector_flow', commit_vector_flow)

---

### Step 2: Launch the Vector Drawing Pad

Run this code in the second cell. This will generate the black background canvas with the white and red pen options.

In [4]:
canvas_html = """
<div id="wrapper" style="background-color: #000; padding: 25px; border-radius: 15px; display: inline-block; color: #00FF00; font-family: 'Courier New', monospace; border: 2px solid #222;">
    <div style="margin-bottom: 15px; letter-spacing: 2px;">VECTOR FLOW SENSOR :: STATUS ACTIVE</div>
    <canvas id="canvas" width="700" height="450" style="border: 1px solid #00FF00; cursor: crosshair; background-color: #000;"></canvas>

    <div style="margin-top: 15px; display: flex; justify-content: space-between; align-items: center;">
        <div>
            <button onclick="changeColor('#FF0000')" style="background: #000; color: #FF0000; border: 2px solid #FF0000; padding: 8px 20px; cursor: pointer; font-weight: bold;">RED PEN</button>
            <button onclick="changeColor('#FFFFFF')" style="background: #000; color: #FFFFFF; border: 2px solid #FFFFFF; padding: 8px 20px; cursor: pointer; font-weight: bold; margin-left: 10px;">WHITE PEN</button>
        </div>

        <button onclick="commitFlow()" style="background: #00FF00; color: #000; border: none; padding: 10px 25px; cursor: pointer; font-weight: bold; font-size: 14px; box-shadow: 0 0 10px #00FF00;">COMMIT TO BLOCKCHAIN</button>
    </div>

    <div id="telemetry" style="font-size: 11px; margin-top: 15px; color: #00AA00; border-top: 1px solid #222; padding-top: 10px;">
        TRANSMISSION READY | VECTORS: 0
    </div>
</div>

<script>
    var canvas = document.getElementById('canvas');
    var ctx = canvas.getContext('2d');
    var drawing = false;
    var color = '#FF0000';
    var currentPath = [];

    // Set initial black void
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    function changeColor(c) { color = c; }

    canvas.addEventListener('mousedown', (e) => {
        drawing = true;
        ctx.beginPath();
        ctx.moveTo(e.offsetX, e.offsetY);
    });

    canvas.addEventListener('mousemove', (e) => {
        if (drawing) {
            // Translate physical movement into vector coordinate data
            currentPath.push({x: e.offsetX, y: e.offsetY, color: color, time: Date.now()});

            ctx.lineTo(e.offsetX, e.offsetY);
            ctx.strokeStyle = color;
            ctx.lineWidth = 3;
            ctx.lineCap = 'round';
            ctx.stroke();

            document.getElementById('telemetry').innerText = "TRANSMITTING DATA | VECTORS CAPTURED: " + currentPath.length;
        }
    });

    canvas.addEventListener('mouseup', () => drawing = false);
    canvas.addEventListener('mouseleave', () => drawing = false);

    async function commitFlow() {
        if (currentPath.length === 0) return;

        document.getElementById('telemetry').innerText = "HASHING VECTOR FLOW...";
        const data = JSON.stringify(currentPath);

        // Invoke the Python blockchain logic
        await google.colab.kernel.invokeFunction('notebook.commit_vector_flow', [data], {});

        // Clear local path for the next block
        currentPath = [];
        document.getElementById('telemetry').innerText = "BLOCKCHAIN SYNC COMPLETE | VECTORS: 0";

        // Optional: Reset canvas view after commit
        // ctx.fillStyle = "black";
        // ctx.fillRect(0, 0, canvas.width, canvas.height);
    }
</script>
"""

display(IPython.display.HTML(canvas_html))

---

### Step 3: Inspect the Ledger

At any point, run this final command to see the translated vector data stored in the chain.

In [7]:
# View the blockchain history
for block in vector_blockchain:
    print(f"Index: {block['index']} | Vectors: {block['vector_points']} | Hash: {block['hash'][:20]}...")

### Step 4: Visualize a Stored Vector Block

This code will take the `vector_data` from the last committed block and plot it using `matplotlib`, displaying the path with its original colors.

In [8]:
import matplotlib.pyplot as plt

if vector_blockchain:
    # Get the last committed block for visualization
    last_block = vector_blockchain[-1]
    vector_data_to_plot = last_block['vector_data']

    # Prepare data for plotting
    x_coords = [point['x'] for point in vector_data_to_plot]
    y_coords = [point['y'] for point in vector_data_to_plot]
    colors = [point['color'] for point in vector_data_to_plot]

    # The canvas y-coordinates are typically inverted in browser (0 at top),
    # but for plotting we might want 0 at bottom. Let's keep original for now.
    # If you want to invert, you can use: max_y - point['y']
    max_y_canvas = 450 # From the canvas HTML
    # y_coords_inverted = [max_y_canvas - y for y in y_coords]

    plt.figure(figsize=(10, 7))
    plt.title(f"Visualization of Vector Block {last_block['index']}")
    plt.xlabel("X-coordinate")
    plt.ylabel("Y-coordinate")

    # Plot each segment with its specific color
    for i in range(len(x_coords) - 1):
        plt.plot(x_coords[i:i+2], y_coords[i:i+2], color=colors[i], linewidth=3)

    plt.scatter(x_coords[0], y_coords[0], color='green', s=100, label='Start', zorder=5)
    plt.scatter(x_coords[-1], y_coords[-1], color='blue', s=100, label='End', zorder=5)

    plt.gca().invert_yaxis() # Invert y-axis to match canvas's top-left origin
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axis('equal') # Ensure x and y scales are the same
    plt.legend()
    plt.show()

else:
    print("No blocks in the blockchain to visualize. Please draw something and commit it first.")

No blocks in the blockchain to visualize. Please draw something and commit it first.


252